# Shopper Demographics Analysis

## Overview
Extract and visualize shopper demographics (gender and age) from Customer IDPOS data.

**Use Cases:**
- Understand customer profile by product/brand
- Identify target demographic segments
- Compare demographic patterns across categories
- Support marketing and targeting strategies

**Data Source:** `cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw`

**Template Updated:** 2025/03/09  
**Python Migration:** 2026/01/07

## 1. Setup and Configuration

In [1]:
# Import required libraries
import databricks.sql as sql
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [4]:
# Load environment variables from parent directory
load_dotenv(dotenv_path='../.env')

# Databricks connection parameters
DATABRICKS_HOST = os.getenv('DATABRICKS_HOST')
DATABRICKS_TOKEN = os.getenv('DATABRICKS_TOKEN')
DATABRICKS_HTTP_PATH = os.getenv('DATABRICKS_HTTP_PATH')

# Validate required parameters
if not all([DATABRICKS_HOST, DATABRICKS_TOKEN, DATABRICKS_HTTP_PATH]):
    raise ValueError("Missing required Databricks credentials. Please check your .env file.")

print(f"✓ Configuration loaded")
print(f"  Host: {DATABRICKS_HOST}")

✓ Configuration loaded
  Host: https://adb-2258763851730787.7.azuredatabricks.net


## 2. Analysis Parameters

Configure the demographic analysis parameters below.

In [19]:
# =============================================================================
# ANALYSIS PARAMETERS
# =============================================================================

# Customer Filter (Data Provider Codes)
# Options: 'cds_8005' (TSURUHA), 'cds_8006' (TOMODS), 'cds_8007' (SAPPORO DRUG),
#          'cds_8008' (KOHNAN), 'cds_8009' (FUJI YAKUHIN), 'cds_8010' (TRIAL),
#          'cds_8011' (CHUBU YAKUHIN), 'cds_8012' (CAINZ), 'cds_8013' (SUGI YAKKYOKU)
customer_filter = 'cds_8005'

# Analysis Period
start_date = '2025-01-01'
end_date = '2025-12-31'

# Category Filter
# Options: 'Air Care', 'Appliances', 'Baby Care', 'Dish Care', 'Fabric Enhancer',
#          'Feminine Care', 'Hair Care', 'Household cleaning', 'Kitchen Cleaning',
#          'Laundry', 'Oral Care', 'Shave Care', 'Super Premium Skin Care'
category_filter = 'Laundry'

# Target Product Condition (WHERE clause format)
# Examples:
#   - jp_sub_brand_alter_lang_name = 'アリエール'
#   - jp_brand_name = 'Ariel'
#   - jp_segment_name IN ('Premium', 'Value')
target_condition = "jp_sub_brand_alter_lang_name = 'ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ'"

print("✓ Parameters configured:")
print(f"  Customer: {customer_filter}")
print(f"  Period: {start_date} to {end_date}")
print(f"  Category: {category_filter}")
print(f"  Target condition: {target_condition}")

✓ Parameters configured:
  Customer: cds_8005
  Period: 2025-01-01 to 2025-12-31
  Category: Laundry
  Target condition: jp_sub_brand_alter_lang_name = 'ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ'


## 3. Build and Execute Query

In [20]:
# Build SQL query
query = f"""
WITH tran_table AS (
    SELECT
        data_provider_key,
        prod_key,
        shopper_key,
        transact_id,
        data_provider_code_part,
        sales_period_group_end_date_part,
        pos_unit_sales_qty,
        pos_sales_amt
    FROM
        cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw
    WHERE
        sales_period_group_end_date_part BETWEEN DATE '{start_date}' AND DATE '{end_date}'
        AND data_provider_code_part = '{customer_filter}'
),
prod_table AS (
    SELECT
        prod_key,
        jp_brand_name,
        jp_mfgr_name,
        jp_category_name,
        jp_sub_category_name,
        jp_sub_category_alter_lang_name,
        jp_sub_segment_name,
        jp_sub_brand_name,
        jp_sub_brand_alter_lang_name,
        jp_prod_name,
        jp_prod_alter_lang_name,
        jp_item_gtin,
        jp_prod_family_1_name,
        jp_prod_form_name,
        jp_segment_1_name
    FROM
        id_pos_ai_1.prod_dim_ext_vw
    WHERE
        jp_category_name = '{category_filter}'
),
shopper_table AS (
    SELECT
        shopper_key,
        member_ind,
        gender_code,
        age
    FROM
        id_pos_ai_1.shopper_dim_generic_vw
    WHERE
        data_provider_code_part = '{customer_filter}'
)
SELECT
    gender_code,
    CASE gender_code
        WHEN '0' THEN 'Not Registered'
        WHEN '1' THEN 'Male'
        WHEN '2' THEN 'Female'
        ELSE 'Unknown'
    END AS gender,
    age,
    COUNT(DISTINCT shopper_table.shopper_key) AS num_of_shopper
FROM
    tran_table
    LEFT OUTER JOIN prod_table ON tran_table.prod_key = prod_table.prod_key
    LEFT OUTER JOIN shopper_table ON tran_table.shopper_key = shopper_table.shopper_key
WHERE
    {target_condition}
    AND shopper_table.shopper_key IS NOT NULL
    AND age BETWEEN 0 AND 100
GROUP BY
    gender_code,
    gender,
    age
ORDER BY
    gender_code,
    age
"""

print("✓ SQL query constructed")
print(f"  Query length: {len(query)} characters")

✓ SQL query constructed
  Query length: 1886 characters


In [21]:
# Execute query
print("Connecting to Databricks...")

with sql.connect(
    server_hostname=DATABRICKS_HOST,
    http_path=DATABRICKS_HTTP_PATH,
    access_token=DATABRICKS_TOKEN
) as connection:
    print("✓ Connected successfully")
    print("Executing query...")
    
    with connection.cursor() as cursor:
        cursor.execute(query)
        
        # Fetch results into DataFrame
        columns = [desc[0] for desc in cursor.description]
        results = cursor.fetchall()
        df = pd.DataFrame(results, columns=columns)
        
        print(f"✓ Query completed successfully")
        print(f"  Retrieved {len(df)} rows")
        print(f"  Age range: {df['age'].min()} - {df['age'].max()}")
        print(f"  Total shoppers: {df['num_of_shopper'].sum():,.0f}")

Connecting to Databricks...
✓ Connected successfully
Executing query...
✓ Query completed successfully
  Retrieved 29 rows
  Age range: 0 - 100
  Total shoppers: 485,817


## 4. Data Processing and Summary

In [22]:
# Convert data types
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df['num_of_shopper'] = pd.to_numeric(df['num_of_shopper'], errors='coerce')

# Create age groups for better visualization
df['age_group'] = pd.cut(df['age'], 
                          bins=[0, 20, 30, 40, 50, 60, 70, 100],
                          labels=['0-20', '21-30', '31-40', '41-50', '51-60', '61-70', '70+'])

# Summary by gender
gender_summary = df.groupby('gender').agg({
    'num_of_shopper': 'sum'
}).reset_index()
gender_summary['percentage'] = (gender_summary['num_of_shopper'] / gender_summary['num_of_shopper'].sum() * 100).round(1)

print("\n" + "="*60)
print("DEMOGRAPHIC SUMMARY")
print("="*60)
print("\nGender Distribution:")
print(gender_summary.to_string(index=False))

# Summary by age group
age_group_summary = df.groupby(['gender', 'age_group']).agg({
    'num_of_shopper': 'sum'
}).reset_index()

print("\nTop 5 Age Groups by Gender:")
top_age_groups = df.nlargest(5, 'num_of_shopper')[['gender', 'age', 'num_of_shopper']]
print(top_age_groups.to_string(index=False))


DEMOGRAPHIC SUMMARY

Gender Distribution:
        gender  num_of_shopper  percentage
        Female          343010        70.6
          Male          108976        22.4
Not Registered           33831         7.0

Top 5 Age Groups by Gender:
gender  age  num_of_shopper
Female   50           80188
Female   40           67231
Female   60           58787
Female   70           45840
Female   30           39191


C:\Users\sugimoto.k.1\AppData\Local\Temp\2\ipykernel_15256\3986938313.py:23: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



## 5. Visualizations

### 5.1 Gender Distribution (Pie Chart)

In [23]:
# Gender distribution pie chart
fig = px.pie(gender_summary, 
             values='num_of_shopper', 
             names='gender',
             title=f'Gender Distribution - {category_filter} Category<br><sub>{start_date} to {end_date}</sub>',
             color='gender',
             color_discrete_map={'Male': '#4A90E2', 'Female': '#E24A90', 'Not Registered': '#CCCCCC'},
             hole=0.4)

fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(height=500, showlegend=True)
fig.show()

print(f"\nTotal Shoppers: {gender_summary['num_of_shopper'].sum():,.0f}")


Total Shoppers: 485,817


### 5.2 Age Distribution by Gender (Population Pyramid)

In [24]:
# Create population pyramid
# Filter out non-registered and prepare data
pyramid_data = df[df['gender'].isin(['Male', 'Female'])].copy()

# Aggregate by age and gender
pyramid_agg = pyramid_data.groupby(['age', 'gender'])['num_of_shopper'].sum().reset_index()

# Separate male and female
male_data = pyramid_agg[pyramid_agg['gender'] == 'Male'].copy()
female_data = pyramid_agg[pyramid_agg['gender'] == 'Female'].copy()

# Make male values negative for pyramid effect
male_data['num_of_shopper'] = -male_data['num_of_shopper']

# Create figure
fig = go.Figure()

# Add male bar
fig.add_trace(go.Bar(
    y=male_data['age'],
    x=male_data['num_of_shopper'],
    name='Male',
    orientation='h',
    marker=dict(color='#4A90E2'),
    hovertemplate='Age: %{y}<br>Male: %{x:,}<extra></extra>'
))

# Add female bar
fig.add_trace(go.Bar(
    y=female_data['age'],
    x=female_data['num_of_shopper'],
    name='Female',
    orientation='h',
    marker=dict(color='#E24A90'),
    hovertemplate='Age: %{y}<br>Female: %{x:,}<extra></extra>'
))

fig.update_layout(
    title=f'Age Distribution by Gender - {category_filter} Category<br><sub>{start_date} to {end_date}</sub>',
    xaxis=dict(title='Number of Shoppers', tickformat=',d'),
    yaxis=dict(title='Age'),
    barmode='overlay',
    bargap=0.1,
    height=600,
    hovermode='y unified'
)

fig.show()

print(f"\nAge range: {pyramid_data['age'].min():.0f} - {pyramid_data['age'].max():.0f} years")
print(f"Average age (Male): {male_data['age'].mean():.1f} years")
print(f"Average age (Female): {female_data['age'].mean():.1f} years")


Age range: 0 - 100 years
Average age (Male): 46.0 years
Average age (Female): 46.0 years


### 5.3 Age Group Distribution (Stacked Bar Chart)

In [25]:
# Age group analysis
fig = px.bar(age_group_summary[age_group_summary['gender'].isin(['Male', 'Female'])], 
             x='age_group', 
             y='num_of_shopper',
             color='gender',
             title=f'Shopper Count by Age Group and Gender<br><sub>{category_filter} - {start_date} to {end_date}</sub>',
             labels={'num_of_shopper': 'Number of Shoppers', 'age_group': 'Age Group'},
             color_discrete_map={'Male': '#4A90E2', 'Female': '#E24A90'},
             barmode='group')

fig.update_layout(height=500, xaxis_tickangle=-45)
fig.show()

# Show top age group
top_age_group = age_group_summary.loc[age_group_summary['num_of_shopper'].idxmax()]
print(f"\nLargest demographic segment: {top_age_group['gender']} {top_age_group['age_group']} ({top_age_group['num_of_shopper']:,.0f} shoppers)")


Largest demographic segment: Female 41-50 (80,188 shoppers)


## 6. Detailed Data Table

In [26]:
# Display top 20 rows
print("\nTop 20 Age/Gender Combinations by Shopper Count:")
print("="*60)
df.nlargest(20, 'num_of_shopper')[['gender', 'age', 'age_group', 'num_of_shopper']]


Top 20 Age/Gender Combinations by Shopper Count:


,gender,age,age_group,num_of_shopper
24,Female,50,41-50,80188
23,Female,40,31-40,67231
25,Female,60,51-60,58787
26,Female,70,61-70,45840
22,Female,30,21-30,39191
14,Male,50,41-50,23248
0,Not Registered,10,0-20,21573
27,Female,80,70+,19592
13,Male,40,31-40,18796
15,Male,60,51-60,18663


## 7. Export Results (Optional)

In [27]:
# Uncomment to export
# output_file = f"demographics_{category_filter}_{customer_filter}_{start_date}_to_{end_date}.xlsx"
# with pd.ExcelWriter(output_file) as writer:
#     df.to_excel(writer, sheet_name='Raw Data', index=False)
#     gender_summary.to_excel(writer, sheet_name='Gender Summary', index=False)
#     age_group_summary.to_excel(writer, sheet_name='Age Group Summary', index=False)
# print(f"✓ Results exported to: {output_file}")

print("Export options available (uncomment code above to use)")

Export options available (uncomment code above to use)
